# Week 4 — Practical Lab: A Complete Data Workflow

So far each tool has been learned on its own. This lab puts them together in the order a real analyst uses them.

**The dataset.** `students.csv` — records for a university's students: their program, city, study habits, attendance, previous GPA and final marks. It is deliberately *messy*, the way real data always is.

**The workflow.** Every data project follows roughly the same six stages, and this lab walks through all of them:

| Stage | What it means | Tools |
|---|---|---|
| 1. Load | Get the data into memory | `read_csv` |
| 2. Inspect | Understand its size, columns and types | `head`, `info`, `describe` |
| 3. Clean | Fix duplicates, inconsistencies, missing values | `drop_duplicates`, `fillna` |
| 4. Organize | Filter, sort, and create useful new columns | masks, `cut`, `apply` |
| 5. Analyze | Answer questions about groups and relationships | `groupby`, `merge`, `corr` |
| 6. Report | State findings in plain language | markdown |

**A note on the order.** It is not arbitrary. You cannot trust an average computed from duplicated rows, and you cannot compute a correlation on a column full of gaps. Cleaning comes before analysis because analysis done on dirty data produces confident, wrong answers.

In [ ]:
import numpy as np
import pandas as pd

---
# Stage 1 — Load the data

`read_csv` reads a comma-separated file into a DataFrame. Both files should sit in the same folder as this notebook.

In [ ]:
df = pd.read_csv("students.csv")
city_info = pd.read_csv("city_info.csv")

print("students :", df.shape)      # (rows, columns)
print("city_info:", city_info.shape)

---
# Stage 2 — Inspect

Before changing anything, look at what you have. Three questions: *how big is it, what are the columns, and what is missing?*

In [ ]:
df.head()

**What this does:** shows the first 5 rows. It is the fastest way to see whether the file loaded correctly, whether the column names came through, and what the values look like.

In [ ]:
df.info()

**What this does:** lists every column with its data type and its count of non-missing values. Any column whose count is below the total row count has gaps. Note `study_hours`, `attendance`, and `prev_gpa` here.

In [ ]:
df.describe().round(2)

**What this does:** the five-number summary plus mean and standard deviation, for every numeric column. Scan it for impossible values — a negative age, an attendance above 100, a GPA above 4.0 would all signal a data-entry problem.

Note the gap between the **mean** and the **50%** (median) of `study_hours`: when the mean sits noticeably above the median, the column is **right-skewed** — a few students study far more than the rest.

In [ ]:
df.isnull().sum()

In [ ]:
# Which columns are missing, as a percentage
(df.isnull().sum() / len(df) * 100).round(1)

**What this does:** quantifies the gaps. Under about 5% is usually easy to fill; a column missing 40%+ may be worth dropping entirely. Here the gaps are small enough to fill.

---
# Stage 3 — Clean

Three problems to fix, in order: duplicated rows, inconsistent text, and missing values.

### 3.1 Duplicate rows

A duplicate is the same record entered twice. It silently biases every average, because one student gets counted twice.

In [ ]:
print("duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("student_id").head(8)

In [ ]:
df = df.drop_duplicates()
print("rows after removing duplicates:", len(df))

**What this does:** `duplicated()` flags rows identical to an earlier row; `drop_duplicates()` removes them, keeping the first occurrence.

### 3.2 Inconsistent categories

Text entered by hand is rarely consistent. To the computer, `"Male"`, `"male"` and `"MALE"` are three different categories.

In [ ]:
print(df["gender"].unique())
print(df["city"].unique())

In [ ]:
# Standardize: strip stray spaces, then apply consistent capitalization
df["gender"] = df["gender"].str.strip().str.title()
df["city"]   = df["city"].str.strip().str.title()

print(df["gender"].unique())
print(df["city"].unique())

**What this does:** `.str.title()` converts each value to Title Case, so `"male"` and `"MALE"` both become `"Male"`. Without this step, a `groupby("gender")` would produce four groups instead of two, and every summary would be wrong.

### 3.3 Missing values

Now decide, column by column, what each gap should become.

In [ ]:
for col in ["study_hours", "attendance", "prev_gpa"]:
    print(f"{col:12s} mean={df[col].mean():6.2f}   median={df[col].median():6.2f}   skew={df[col].skew():5.2f}")

In [ ]:
# Skewed column → median.  Roughly symmetric column → mean is acceptable, median is still safe.
df["study_hours"] = df["study_hours"].fillna(df["study_hours"].median())
df["attendance"]  = df["attendance"].fillna(df["attendance"].median())
df["prev_gpa"]    = df["prev_gpa"].fillna(df["prev_gpa"].median())

print("remaining missing values:", df.isnull().sum().sum())

**What this does, and why the median.** `study_hours` is right-skewed — a handful of very high values drag the mean upward, so the mean is not a typical student. The median is unaffected by those extremes, which makes it the safer substitute.

**The honest caveat:** filling gaps is a decision, not a fact. Every filled cell is a guess. It pulls values toward the centre and slightly *reduces* the real spread of the column, so a standard deviation computed after filling is a little smaller than the truth. That is an acceptable trade for keeping the rows — but it must be a conscious choice, and it should be written down.

---
# Stage 4 — Organize

The data is now trustworthy. Next: filter it, sort it, and add columns that make the analysis easier.

In [ ]:
# Filtering with a boolean mask — high achievers
high = df[df["final_marks"] > 80]
print(f"{len(high)} students scored above 80 ({len(high)/len(df)*100:.1f}% of the class)")
high[["student_id", "program", "study_hours", "final_marks"]].head()

In [ ]:
# Multiple conditions — each wrapped in its own parentheses
focused = df[(df["study_hours"] > 10) & (df["attendance"] > 85)]
print("students who study a lot AND attend regularly:", len(focused))

In [ ]:
# Sorting — top 5 performers
df.sort_values("final_marks", ascending=False).head(5)[
    ["student_id", "program", "city", "study_hours", "final_marks"]
]

### Creating new columns

Two ways to derive a new column: **binning** a continuous value into bands, and **applying** a function to compute one.

In [ ]:
# Binning: continuous marks → an ordered grade category
df["grade"] = pd.cut(
    df["final_marks"],
    bins=[0, 50, 60, 70, 80, 100],
    labels=["F", "D", "C", "B", "A"]
)
df[["final_marks", "grade"]].head()

In [ ]:
df["grade"].value_counts().sort_index()

**What this does:** `pd.cut` assigns each mark to a band. It converts a *quantitative continuous* column into a *qualitative ordinal* one — the grades have a natural order (A > B > C). This is the conversion from the data-types lesson, applied for real.

In [ ]:
# Applying a function to compute a new column
df["study_level"] = df["study_hours"].apply(
    lambda h: "high" if h >= 10 else ("medium" if h >= 5 else "low")
)
df[["study_hours", "study_level"]].head()

---
# Stage 5 — Analyze

Now the actual questions. Each one is a single line of Pandas.

### 5.1 Group comparisons

**Question: does the amount of studying show up in the marks?**

In [ ]:
df.groupby("study_level", observed=True)["final_marks"].mean().round(2)

In [ ]:
# The fuller picture: how many students, and how much variation, in each group
df.groupby("study_level", observed=True)["final_marks"].agg(
    ["count", "mean", "std", "min", "max"]
).round(2)

**What this does:** splits students by study level, then computes several statistics for each group. The `count` column matters — a group's mean is unreliable if only a few students are in it. The `std` shows how consistent each group is: a high standard deviation means students in that band vary widely despite similar study habits.

In [ ]:
# Question: how does performance differ across programs?
df.groupby("program")["final_marks"].agg(["count", "mean", "std"]).round(2).sort_values("mean", ascending=False)

In [ ]:
# Grouping by two columns at once
df.groupby(["program", "gender"])["final_marks"].mean().round(2)

In [ ]:
# Scholarship rate by program — the mean of a 0/1 column is a proportion
(df.groupby("program")["scholarship"].mean() * 100).round(1)

**What this does:** because `scholarship` holds only 0 and 1, its mean is the fraction of 1s — the scholarship rate. Multiplying by 100 turns it into a percentage. This trick works for any yes/no column stored as 0/1.

### 5.2 Merging in the lookup table

The student records hold a city name but nothing about that city. A second table holds the province and campus type. Merging attaches that information to every student.

In [ ]:
city_info

In [ ]:
df = pd.merge(df, city_info, on="city", how="left")
df[["student_id", "city", "province", "campus_type"]].head()

**What this does, and why `left`.** A left join keeps **every** student row and attaches city information where it matches. Had we used `inner`, any student from a city missing from the lookup table would silently vanish from the analysis. When you have a main table and are adding reference information to it, `left` is almost always the right choice.

Always verify a merge did what you expected:

In [ ]:
print("rows after merge:", len(df))                       # should be unchanged
print("unmatched cities:", df['province'].isnull().sum())  # should be 0

In [ ]:
# Now a question we could not previously ask
df.groupby("province")["final_marks"].agg(["count", "mean"]).round(2).sort_values("mean", ascending=False)

In [ ]:
df.groupby("campus_type")["final_marks"].mean().round(2)

### 5.3 Correlation — which factors track final marks?

In [ ]:
numeric = ["study_hours", "attendance", "prev_gpa", "final_marks"]
df[numeric].corr().round(3)

In [ ]:
# Just the column we care about, sorted
df[numeric].corr()["final_marks"].drop("final_marks").sort_values(ascending=False).round(3)

**How to read this.** Each number is between −1 and +1 and measures how strongly that column moves together with `final_marks`. Study hours shows the strongest relationship, attendance a moderate one, previous GPA a weaker one.

**And the essential caution:** a strong correlation does **not** prove that studying *causes* higher marks. Motivated students may both study more *and* attend more *and* score higher — motivation would be the hidden cause behind all three. Correlation identifies a pattern worth investigating; it never settles the question of cause.

In [ ]:
# Correlations can also be computed within a group
df.groupby("program", observed=True)[["study_hours", "final_marks"]].corr().round(3).iloc[0::2, 1]

---
# Stage 6 — Report

An analysis nobody can read has no value. Close every project by stating what you found in plain sentences, with the numbers that support each claim.

Run the cell below to gather the key figures, then write your findings underneath.

In [ ]:
print("FINAL DATASET")
print(f"  students: {len(df)}   columns: {df.shape[1]}   missing values: {df.isnull().sum().sum()}")
print()
print("MARKS")
print(f"  mean {df['final_marks'].mean():.1f}   median {df['final_marks'].median():.1f}   std {df['final_marks'].std():.1f}")
print()
print("BY STUDY LEVEL")
print(df.groupby("study_level", observed=True)["final_marks"].mean().round(1).to_string())
print()
print("STRONGEST DRIVER OF MARKS")
c = df[numeric].corr()["final_marks"].drop("final_marks")
print(f"  {c.idxmax()} (r = {c.max():.3f})")

### Your findings

*(Write 3–5 bullet points here in plain English. Each should state something you found and the number that backs it up. Example: "Students in the 'high' study group averaged X marks, compared with Y for the 'low' group — a difference of Z.")*

1. 
2. 
3. 

---
## What this workflow accomplished

Starting from a messy file, we:

- **removed** duplicated records that would have double-counted students,
- **standardized** inconsistent category text so grouping produced correct counts,
- **filled** missing values with the median, keeping every row while recording the trade-off,
- **derived** grade bands and study levels that made comparison possible,
- **grouped** to compare programs, genders, provinces and study habits,
- **merged** external reference data to unlock questions the original file could not answer,
- **measured** which factors track final marks — while refusing to claim causation.

Every one of these steps used a tool from this week. The tools are not the point; **the order is.** Clean before you analyze, verify after you merge, and always state your findings in words a non-programmer could follow.